# Recruitment Data — Cleaning & KPIs (template)

> Runs on **synthetic data** only. Generate it first with `python generate_synthetic_data.py`.

In [ ]:
import pandas as pd

df_raw = pd.read_csv('synthetic/interviews.csv', parse_dates=['interview_date', 'offer_date', 'acceptance_date'])
df_raw.shape

## 1. Profiling

In [ ]:
df_raw.info()
df_raw.isna().mean().sort_values(ascending=False).round(3)

## 2. Cleaning

Each step is logged (see `docs/methodology.md`).

In [ ]:
log = []

def step(df, name, func):
    before = len(df)
    out = func(df)
    log.append({'step': name, 'rows_before': before, 'rows_after': len(out)})
    return out

df = df_raw.copy()
df = step(df, 'drop exact duplicates', lambda d: d.drop_duplicates())
df = step(df, 'normalize text columns', lambda d: d.assign(**{c: d[c].str.strip() for c in d.select_dtypes(include=['object', 'string']).columns}))
df = step(df, 'remove inconsistent dates', lambda d: d[~(d['offer_date'] < d['interview_date'])])
pd.DataFrame(log)

## 3. KPIs

In [ ]:
done = df[df['interview_status'] == 'done']
kpis = {
    'interviews': len(done),
    'offers': int(df['offer_date'].notna().sum()),
    'hires': int((df['outcome'] == 'hired').sum()),
}
kpis['interview_to_offer_rate'] = round(kpis['offers'] / kpis['interviews'], 3)
kpis['offer_acceptance_rate'] = round(kpis['hires'] / kpis['offers'], 3)
kpis['median_days_to_offer'] = (df['offer_date'] - df['interview_date']).dt.days.median()
kpis

## 4. Breakdowns

In [ ]:
# Interviews per week
weekly = done.set_index('interview_date').resample('W')['interview_id'].count()
weekly.plot(title='Interviews per week');

In [ ]:
# Average score by school group (n >= 10)
g = done.groupby('school_group')['score'].agg(['mean', 'count'])
g[g['count'] >= 10].sort_values('mean', ascending=False)

In [ ]:
# Conversion by source
conv = df.groupby('source').agg(applications=('candidate_id', 'count'), hires=('outcome', lambda s: (s == 'hired').sum()))
conv['conversion_rate'] = (conv['hires'] / conv['applications']).round(3)
conv.sort_values('conversion_rate', ascending=False)

In [ ]:
# Top dropout reasons
df.loc[df['outcome'] == 'dropout', 'outcome_reason'].value_counts()

## 5. Insights

| # | Insight | Evidence | Limits | Recommendation |
|---|---|---|---|---|
| 1 | | | | |